# Publish to Delta Lake

## Objective

Send a completed run to Delta tables and safely repeat the publication.

Run notebooks 2 through 4 first. This writes saved forecasts and scores, including every retained tuning window, without fitting or rescoring. You need write access to your chosen destination. These adapters publish results; they do not add source-data ingestion from this database.

Every row carries study ID/hash, workflow run ID, stage, origin and horizon. Tuning rows also carry search ID, trial ID, attempt ID and zero-based window index. Final validation/test rows have null tuning identifiers. Join using the complete identity and null-safe comparisons. Recorded scores remain in `evidence_json`.

Delta is a table format built on Parquet and a transaction log. It is not limited to Databricks. This local example needs Java 17 and the Delta runtime matching Spark 4.2. Start a fresh kernel; Spark extensions must be set before the session starts.


## Step 1: Choose a place for the example files

We will keep data and results outside the library folder. The printed path shows where to find them. Set the environment variable only if you want a different folder.


We also turn on progress messages, so we can see which step is running.


In [ ]:
# Import the tools used in this step.
import os
from pathlib import Path

from autoforecast import configure_logging

# Choose the workspace without changing the library checkout.
work = (
    Path(os.environ.get("AUTOFORECAST_TUTORIAL_DIR", Path.home() / "autoforecast-tutorials"))
    .expanduser()
    .resolve()
)
# Create the folder if this is your first visit.
work.mkdir(parents=True, exist_ok=True)
print(work)

# Show the current step while the library is working.
configure_logging()


### Understand the output

You should see the full workspace path. Open that folder to find the files created in these lessons. This path is a folder, not a forecast result.


Turning logging on does not run a forecast. Later library calls will show their steps. A stage count tells us how many steps have finished, not how much time is left.


## Step 2: Start Spark with Delta enabled

We will load the Delta extension before starting Spark. This must run in a fresh kernel so an older Spark session cannot hide the new settings.

We will also keep a local catalog, which remembers the table names after you restart the kernel. Its files and log stay in the workspace beside the Delta data.

Only one kernel can use this local catalog at a time. Shut down any earlier kernel for this lesson in Jupyter’s Running panel before starting this cell.


In [ ]:
# Import the tools used in this step.
import shlex

from pyspark.sql import SparkSession

# Keep the table catalog and its log in the workspace so tables reopen next time.
catalog_home = work / "delta-catalog"
catalog_home.mkdir(exist_ok=True)
# Quote each Java setting safely, including workspace paths with spaces.
java_options = shlex.join([
    f"-Dderby.system.home={catalog_home}",
    f"-Dderby.stream.error.file={work / 'delta-derby.log'}",
])

# Use two local workers for this small example.
spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.jars.packages", "io.delta:delta-spark_4.2_2.13:4.4.1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", str(work / "delta-warehouse"))
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        "jdbc:derby:metastore_db;create=true",
    )
    .config("spark.driver.extraJavaOptions", java_options)
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

### Understand the output

Spark should start with the Delta extension. The first run may download runtime files. If it cannot load Delta, check the package version and restart the kernel before retrying.


## Step 3: Open the registry and saved run

We will reuse the file-backed registry and load the completed workflow. The destination tables will hold a copy of its results.


In [ ]:
# Import the tools used in this step.
from autoforecast.study_registration import StudyRegistry
from autoforecast.study_workflow import load_forecast_workflow

# Open the registry described by our saved study settings.
registry = StudyRegistry.from_config(work / "study.yaml", spark=spark)

# Check and open the saved run without fitting again.
result = load_forecast_workflow((work / "latest-run.txt").read_text().strip())

### Understand the output

You may see verification messages while the saved files are checked. The registry and completed run are ready; this cell has not written database results.


## Step 4: Create destination tables and publish

We will create the example namespace and tables, publish the run twice, and list its result rows. The second call should reuse the same rows.


In [ ]:
# Import the Delta destination already used by the study registry.
from autoforecast.integrations.delta import DeltaStudyStore

# Prepare missing example tables, using one shared lock for initial setup.
destination = DeltaStudyStore(spark, namespace="autoforecast_tutorial")
destination.prepare(lock_path=work / "delta-setup.lock")
# Publish twice, checking only this study and workflow each time.
for attempt in range(2):
    publication = registry.publish_results(result.study_id, result.path, destination=destination)
    scope = {
        "study_id": result.study_id,
        "study_sha": publication["study_sha256"],
        "run_id": result.run_id,
    }
    rows, _ = destination.read("study_results", scope)
    forecasts, _ = destination.read("forecasts", scope)
    print(
        "Publication",
        attempt + 1,
        publication["status"],
        "result rows:",
        rows.count(),
        "forecast rows:",
        forecasts.count(),
    )
# Include the trial and window columns so earlier fits remain distinguishable.
rows.select("stage", "trial_id", "window_index", "origin", "horizon").orderBy(
    "stage", "trial_id", "window_index", "origin"
).show(truncate=False)

### Understand the output

Each publication should report `published`, 6 result rows and 84 forecast rows for this run. The second call should show the same counts, even if older runs are in the tables. The preview contains four tuning fits, validation and test. `trial_id` distinguishes the two tuning settings, and `window_index` is 0 or 1 for their earlier windows. These fields are empty (`null` or `None`) for validation and test. `origin` is the last training date; `horizon: 7` means seven future days. These six result rows summarize fits; the 84 forecast rows hold individual product/date predictions.


At INFO level, stage messages and a progress panel also show the work as it happens. The publication status and row counts described above is the result to keep. A completed-stage percentage counts steps; it does not predict how much time remains.


Delta writes forecasts first, then result rows that mark a published run. This is not one transaction across both tables. Retry after failure using the same run. The study registry and model artifacts still need supported local or mounted filesystem storage.

The example provisions five tables through the existing Delta store, but this explicit destination call publishes result/forecast tables only. A Delta-backed registry also stores study definitions, frozen observations and data issues.

Existing destinations must have the exact current schema. Use new result/forecast tables to backfill older workflows whose published rows differ. AutoForecast does not silently migrate or overwrite them. Inspect the schema in `autoforecast.integrations.delta.SCHEMAS` before an explicit migration.


## Step 5: Close this Spark session

We are finished with Spark in this notebook. Closing it frees resources before the next lesson.


In [ ]:
# Release Spark resources before moving to another notebook.
spark.stop()

### Understand the output

No output is expected. Spark workers have stopped, but this kernel still holds the local catalog open. In Jupyter’s Running panel, shut down this notebook’s kernel before opening this lesson in another kernel. Saved tables and forecast files remain on disk.
